In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/hotel_bookings.csv")

df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


# Data cleaning

This notebook is used to clean and prepare the hotel booking dataset for modelling.

The main goals of this stage are to:
- review data quality issues identified during EDA.
- decide how to handle missing values.
- remove or treat unrealistic records.
- address duplicate rows.
- prepare a cleaned dataset for modelling.

In [2]:
df_clean = df.copy()

df_clean.shape

(119390, 32)

In [3]:
df_clean.isnull().sum().sort_values(ascending=False)

company                           112593
agent                              16340
country                              488
children                               4
arrival_date_month                     0
arrival_date_week_number               0
hotel                                  0
is_canceled                            0
stays_in_weekend_nights                0
arrival_date_day_of_month              0
adults                                 0
stays_in_week_nights                   0
babies                                 0
meal                                   0
lead_time                              0
arrival_date_year                      0
distribution_channel                   0
market_segment                         0
previous_bookings_not_canceled         0
is_repeated_guest                      0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
previous_cancellations                 0
deposit_type    

## Missing value plan

The main missing value issues in the dataset are:

- `company`
- `agent`
- `country`
- `children`

Initial cleaning approach:
- `children` will likely be filled or reviewed carefully because it has a very small number of missing values.
- `country` might be filled with an `Unknown` category if needed.
- `agent` and `company` have a large number of missing values, so they will need more careful consideration before deciding whether to fill, keep, or drop them.

In [4]:
df_clean[df_clean["children"].isnull()]

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
40600,City Hotel,1,2,2015,August,32,3,1,0,2,...,No Deposit,NaN,NaN,0,Transient-Party,12.0,0,1,Canceled,2015-08-01
40667,City Hotel,1,1,2015,August,32,5,0,2,2,...,No Deposit,14.0,NaN,0,Transient-Party,12.0,0,1,Canceled,2015-08-04
40679,City Hotel,1,1,2015,August,32,5,0,2,3,...,No Deposit,NaN,NaN,0,Transient-Party,18.0,0,2,Canceled,2015-08-04
41160,City Hotel,1,8,2015,August,33,13,2,5,2,...,No Deposit,9.0,NaN,0,Transient-Party,76.5,0,1,Canceled,2015-08-09


In [5]:
df_clean["children"] = df_clean["children"].fillna(0)

df_clean["children"].isnull().sum()

np.int64(0)

### Handling missing values in `children`

The `children` column originally had 4 missing values.

After inspecting the above rows, the records appear valid and include adult guests, with no clear sign that the rows themselves are corrupted. Because the number of missing values was very small, and the missing entries most likely represent no children being included in the booking, these values were filled with `0`.

This keeps the rows in the dataset while applying a reasonable assumption.

In [6]:
df_clean["country"].isnull().sum()

np.int64(488)

In [7]:
df_clean["country"] = df_clean["country"].fillna("Unknown")

df_clean["country"].isnull().sum()

np.int64(0)

### Handling missing values in `country`

The `country` column originally had 488 missing values.

Because `country` is a categorical feature, and there is no reliable way to infer the correct missing country, these values were filled with `"Unknown"`.

This keeps the affected rows in the dataset while preserving the information that the original country value was missing.

In [8]:
df_clean["agent"].isnull().sum()

np.int64(16340)

In [9]:
df_clean["agent"].nunique()

333

In [10]:
df_clean["company"].isnull().sum()

np.int64(112593)

In [11]:
df_clean["company"].nunique()

352

### Review of `agent` and `company`

The `agent` and `company` columns both contain a large amount of missing data and some unique values:

- `agent` has 16,340 missing values and 333 unique non null values.
- `company` has 112,593 missing values and 352 unique non null values.

This shows that both columns have a large number of unique values, and `company` in particular has very little usable data. At this stage, both columns should be handled carefully. `company` is likely to be removed, while `agent` may either be dropped or grouped later depending on the needs of the model.

In [12]:
df_clean.isnull().sum().sort_values(ascending=False).head(10)

company                     112593
agent                        16340
hotel                            0
is_canceled                      0
arrival_date_month               0
arrival_date_week_number         0
lead_time                        0
arrival_date_year                0
stays_in_week_nights             0
adults                           0
dtype: int64

### Missing value progress update

After the first round of cleaning:

- missing values in `children` were filled with `0`
- missing values in `country` were filled with `"Unknown"`

The remaining major missing value issues are now:

- `agent`
- `company`
